# Troubleshooting eCAT Workflows

Use this notebook when loading, plotting, units, references, normalization, or peak detection does not behave as expected. It uses the packaged example data so you can separate package/setup issues from data-specific issues.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def display_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return path.name

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import ecat as e

DATA_DIR = ROOT / "examples" / "data" / "fe_phoh_cv"
EXPORT_DIR = ROOT / "notebooks" / "_outputs"
EXPORT_DIR.mkdir(exist_ok=True)

e.plotting_style("notebook")
print("eCAT version:", getattr(e, "__version__", "unknown"))
print("Example data:", display_path(DATA_DIR))
print("Text files:", len(list(DATA_DIR.glob("*.txt"))))

eCAT version: 0.1.0b2
Example data: examples/data/fe_phoh_cv
Text files: 13


In [2]:
cvs = e.get_data({
    "folder path": str(DATA_DIR),
    "recursive search": False,
    "reference mode": "none",
    "print": False,
})
cvs = e.filter(cvs, {"segments": 3}, {"print": False})
print(f"Loaded {len(cvs)} CV objects")
print(sorted({type(obj).__name__ for obj in cvs}))

Searching exclusively through:
 examples/data/fe_phoh_cv
13 .txt files found.

Loaded 12 CV objects
['cv']


In [3]:
print("If this loads 10 CV objects, your basic import and parser path works.")
print("Loaded:", len(cvs))
print("Types:", sorted({type(obj).__name__ for obj in cvs}))

If this loads 10 CV objects, your basic import and parser path works.
Loaded: 12
Types: ['cv']


In [4]:
# Check what eCAT parsed from filenames.
parsed = pd.DataFrame([
    {
        "name": obj.name,
        "gas": getattr(obj, "gas", None),
        "solvent": getattr(obj, "solvent", None),
        "scan_rate": getattr(obj, "scan_rate", None),
        "scan_window": getattr(obj, "scan_window", None),
        "compounds": ", ".join(map(str, getattr(obj, "compounds", []))),
    }
    for obj in cvs
])
parsed

,name,gas,solvent,scan_rate,scan_window,compounds
0,MeCN_Ar_0.1MTBAPF6_-1.2_to_1V_100mVs,Ar,MeCN,0.100,None,TBAPF6
1,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_t...,Ar,MeCN,0.100,None,"TBAPF6, Fc, Fe-tpyPY2Me"
2,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_t...,Ar,MeCN,0.100,None,"TBAPF6, Fc, Fe-tpyPY2Me"
3,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_t...,Ar,MeCN,0.025,None,"TBAPF6, Fc, Fe-tpyPY2Me"
4,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_t...,Ar,MeCN,0.500,None,"TBAPF6, Fc, Fe-tpyPY2Me"
5,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_t...,Ar,MeCN,1.000,None,"TBAPF6, Fc, Fe-tpyPY2Me"
6,MeCN_Ar_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.7_t...,Ar,MeCN,0.050,None,"TBAPF6, Fc, Fe-tpyPY2Me"
7,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_-1.2_...,CO2,MeCN,0.100,None,"TBAPF6, Fc, Fe-tpyPY2Me"
8,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_100mM...,CO2,MeCN,0.100,None,"TBAPF6, Fc, Fe-tpyPY2Me, PhOH"
9,MeCN_CO2_0.1MTBAPF6_3mMFc_1mMFe-tpyPY2Me_560mM...,CO2,MeCN,0.100,None,"TBAPF6, Fc, Fe-tpyPY2Me, PhOH"


In [5]:
# Inspect options by function. Category-sorted tables make it easier to find relevant knobs.
e.describe_options("get_data", {"pretty print": False, "return": True}).head(20)

            Category                Option                                                        Default                 Type                           Choices                                                                                                                              Description
          Data/input               columns                                                              3                  int                                                                                                                         Number of columns expected in imported data files.
          Data/input             compounds                                                           None       object or None                                                                                                                 Compound names associated with the electrochemical object.
          Data/input         custom reader                                                           None 

,Category,Option,Default,Type,Choices,Description
0,Data/input,columns,3,int,,Number of columns expected in imported data fi...
1,Data/input,compounds,None,object or None,,Compound names associated with the electrochem...
2,Data/input,custom reader,None,object or None,,User-provided file reader for custom import fo...
3,Data/input,decimal,.,str,,Decimal separator used in imported text files.
4,Data/input,delimiter,",",str,,Column delimiter used in imported text files.
5,Data/input,experiment type,None,str or None,,Experiment type to require or assign. If omitt...
6,Data/input,folder path,.,str,,Folder path searched for electrochemical data ...
7,Data/input,gas,None,str or None,,Gas condition metadata for the electrochemical...
8,Data/input,name alterations,None,object or None,,Filename text replacements applied during import.
9,Data/input,recursive search,True,bool,,Recursively search subfolders during import.


In [6]:
# Missing peak troubleshooting: look at detected peak-related options.
e.describe_options("peak_current", {"pretty print": False, "return": True})

        Category             Option         Default                                               Type                                                                                                                                                                                                                              Description
Fitting/analysis  percent threshold             NaN                                      float or None                                                                                                                                                                                     Percent threshold used in peak or tangent selection.
Fitting/analysis tangent min points             NaN                                        int or None                                                                                                                                  Minimum points for tangent fitting; when omitted, eCAT derives a minimum from the pre-peak data 

,Category,Option,Default,Type,Description
0,Fitting/analysis,percent threshold,NaN,float or None,Percent threshold used in peak or tangent sele...
1,Fitting/analysis,tangent min points,NaN,int or None,Minimum points for tangent fitting; when omitt...
2,Fitting/analysis,tangent potential,NaN,float or None,"Manual tangent anchor potential; when omitted,..."
3,Fitting/analysis,tangent range,auto,"str or float or list[float] or tuple[float, fl...",'auto' chooses a pre-peak baseline region from...
4,Advanced,peak fallback,highest current,str or None,Fallback used by peak_current when no local pe...


In [7]:
# Reference-shift troubleshooting. This tutorial subset is unreferenced by design.
e.describe_options("get_data", {"pretty print": False, "return": True}).query("Category == 'Reference/correction'")

            Category                Option                                                        Default                 Type                           Choices                                                                                                                              Description
          Data/input               columns                                                              3                  int                                                                                                                         Number of columns expected in imported data files.
          Data/input             compounds                                                           None       object or None                                                                                                                 Compound names associated with the electrochemical object.
          Data/input         custom reader                                                           None 

,Category,Option,Default,Type,Choices,Description
14,Reference/correction,allow self reference,True,bool,,Allow a CV to be considered as its own referen...
15,Reference/correction,peak prominence,None,float or None,,Minimum peak prominence for automatic referenc...
16,Reference/correction,reference file,None,str or None,,Explicit reference file used by reference mode...
17,Reference/correction,reference guess,auto,float or str or None,,'auto' locates the reference wave automaticall...
18,Reference/correction,reference keyword,None,str or None,,Single keyword used by reference mode 'keyword...
19,Reference/correction,reference keywords,"[Fc, ferrocene, decamethylferrocene, DmFc, cob...",list[str] or None,,Reference keywords tried by reference mode 'au...
20,Reference/correction,reference label,Fc/Fc+,str,,Axis label used after reference shifting.
21,Reference/correction,reference map,None,dict or None,,Explicit target-to-reference object index mapp...
22,Reference/correction,reference mode,auto,str,"auto, manual, keyword, file, none",'auto' searches imported files with reference ...
23,Reference/correction,reference offset,None,float or None,,Manual reference potential offset.


In [8]:
# Small reproducibility report to paste into an issue/email.
report = {
    "ecat_version": getattr(e, "__version__", "unknown"),
    "data_dir": display_path(DATA_DIR),
    "n_files": len(list(DATA_DIR.glob("*.txt"))),
    "n_loaded": len(cvs),
    "object_types": sorted({type(obj).__name__ for obj in cvs}),
}
report

{'ecat_version': '0.1.0b2',
 'data_dir': 'examples/data/fe_phoh_cv',
 'n_files': 13,
 'n_loaded': 12,
 'object_types': ['cv']}